# 22-32 · Параметризация против инъекции

Практика к разделу [«Основы безопасности веб-приложений»](../../site/chapters/glava-22/22-32-bezopasnost.html).

## Цель

На настоящем sqlite3 сравнить опасную сборку запроса через f-строку с безопасным параметризованным запросом — и увидеть, к чему приводит SQL-инъекция, если её не остановить.

## Рабочий пример — уязвимый вариант

In [1]:
import sqlite3

baza = sqlite3.connect(":memory:")
baza.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
baza.execute("INSERT INTO tasks (title) VALUES ('Обычная задача')")
baza.commit()

vvod_polzovatelya = "x'; DROP TABLE tasks; --"

# ТАК ДЕЛАТЬ НЕЛЬЗЯ — показано специально, чтобы увидеть последствия:
try:
    baza.executescript(f"SELECT * FROM tasks WHERE title = '{vvod_polzovatelya}'")
except sqlite3.Error as oshibka:
    print("sqlite3 сообщил об ошибке:", oshibka)

tablicy_posle = baza.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Таблицы после уязвимого запроса:", tablicy_posle)

Таблицы после уязвимого запроса: []


## Проверка результата — таблица пострадала

In [2]:
nazvaniya_tablic = [t[0] for t in tablicy_posle]
assert "tasks" not in nazvaniya_tablic, "уязвимый вариант удалил таблицу — именно поэтому f-строки в SQL опасны"
print("Так и есть: таблица tasks исчезла — вставленный текст выполнился как часть SQL-команды.")

Так и есть: таблица tasks исчезла — вставленный текст выполнился как часть SQL-команды.


## Эксперимент — параметризованный запрос безопасен

In [3]:
baza2 = sqlite3.connect(":memory:")
baza2.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
baza2.execute("INSERT INTO tasks (title) VALUES ('Обычная задача')")
baza2.commit()

vvod_kak_dannye = "x'; DROP TABLE tasks; --"
rezultat = baza2.execute("SELECT * FROM tasks WHERE title = ?", (vvod_kak_dannye,)).fetchall()

tablicy_posle2 = baza2.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Найдено строк:", rezultat)
print("Таблицы:", tablicy_posle2)

assert [t[0] for t in tablicy_posle2] == ["tasks"]
print("Верно: параметризованный запрос обработал вредоносный текст как обычное значение — таблица цела.")

Найдено строк: []
Таблицы: [('tasks',)]
Верно: параметризованный запрос обработал вредоносный текст как обычное значение — таблица цела.
